https://python.langchain.com/api_reference/google_genai/

In [ ]:
%pip install -U langmem langgraph langchain-google-genai geopy requests

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.0/67.0 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.4/47.4 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.8/64.8 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 21.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.4/70.4 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.8/43.8 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 292.8/292.8 kB 19.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.5/216.5 kB 15.9 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.3
    Uninstalling requests-2.32.3:
      Successful

https://python.langchain.com/docs/integrations/chat/google_vertex_ai_palm/

In [ ]:
!pip install -U langchain-google-vertexai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.0/101.0 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.1/42.1 MB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.7/44.7 kB 1.8 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0


https://python.langchain.com/docs/integrations/providers/huggingface/

In [ ]:
!pip install langchain_huggingface

In [ ]:
from google.colab import userdata
api_key = userdata.get('GOOGLE_API_KEY')
project_id = 'vital-future-460019-g9'

In [ ]:
from langgraph.prebuilt import create_react_agent
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.store.memory import InMemoryStore
from google.colab import userdata
from google.colab import auth
import vertexai

In [ ]:
# InMemorySaver -> Checkpoints are persisted and can be used to restore the state of a thread at a later time.
memory_saver = InMemorySaver()

# InMemoryStore -> The underlying dictionary that stores the key-value pairs.
# https://python.langchain.com/api_reference/core/stores/langchain_core.stores.InMemoryStore.html
memory_store = InMemoryStore()

In [ ]:
try:
    auth.authenticate_user()
    print("User authenticated successfully.")

except Exception as e:
    print(f"Authentication failed: {e}")
    print("Please ensure you are running in a Colab environment and have the necessary permissions.")

if project_id is None:
    raise ValueError("Please set your Google Cloud project ID in Colab userdata as 'YOUR_PROJECT_ID'")

User authenticated successfully.


https://cloud.google.com/python/docs/reference/vertexai/latest



In [ ]:
try:
    vertexai.init(project=project_id, location='us-central1')
    print(f"Vertex AI initialized for project: {project_id}")

except Exception as e:
    print(f"Vertex AI initialization failed: {e}")


Vertex AI initialized for project: vital-future-460019-g9


In [ ]:
MODEL = "gemini-2.0-flash-lite"

agent_model = create_react_agent(
    model=MODEL,
    tools=[],
    store=memory_store,
    checkpointer=memory_saver,
)

In [ ]:
def chat(agent, txt, thread_id):
    result_state = agent.invoke(
        {"messages": [
            {
                "role": "user", "content": txt
            }
          ]
        },
        config={"configurable": {"thread_id": thread_id}}
    )

    return result_state["messages"][-1].content

In [ ]:
thread_1 = "thread-1"
print(chat(agent_model,
"""
Hi there, could you guide me through making traditional Thai Pad Thai at home? I want it to be as authentic as possible.
""",
           thread_1))


In [ ]:
from langmem import create_manage_memory_tool, create_search_memory_tool

